### Imports

In [1]:
import os
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models
from tifffile import TiffFile
from skimage.transform import resize
import scanpy as sc
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# #############################################
# Device
# #############################################
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

Using device: cpu


### Device Configuration

In [2]:
# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

# Base paths
base_dir = Path("C:/Users/mills/Desktop/Angione Lab/OC")
adata_path = base_dir / "OC_adata_hvg.h5ad"
output_dir = base_dir / "spatial"
output_dir.mkdir(exist_ok=True)

# Z-slices
DAPI_FOLDER = base_dir / "DAPI"
Z_SLICE_FILES = [DAPI_FOLDER / f"HumanOvarianCancerPatient2Slice2_images_mosaic_DAPI_z{i}.tif" for i in range(7)]

# Output paths
EMBEDDINGS_SAVE_PATH = output_dir / "OC_adata_hvg_cnn_embeddings.npy"
ADATA_SAVE_PATH = output_dir / "OC_adata_hvg_with_cnn.h5ad"

Using device: cpu


### Paths and Data Setup

In [3]:
adata_hvg = sc.read_h5ad(adata_path)
print("Number of cells:", adata_hvg.n_obs)

Number of cells: 63072


### Patch Extraction Utilities 

In [4]:
patch_size = 64
microns_per_pixel = 0.25

In [5]:
def microns_to_pixels(x, y, microns_per_pixel=0.25):
    return int(x / microns_per_pixel), int(y / microns_per_pixel)

In [6]:
def extract_patch(slice_path, x, y, patch_size=64):
    half = patch_size // 2
    with TiffFile(slice_path) as tif:
        img = tif.asarray(out='memmap')
        h, w = img.shape
        x1, y1 = max(x-half, 0), max(y-half, 0)
        x2, y2 = min(x+half, w), min(y+half, h)
        patch = img[y1:y2, x1:x2].astype(np.float32)
        patch = np.clip(patch, 0, None)
        if np.max(patch) > 0:
            patch = (patch - np.min(patch)) / (np.max(patch) - np.min(patch) + 1e-6)
        else:
            patch[:] = 0.0
    return patch

### CNN Model (ResNet18) Setup

In [7]:
resnet = models.resnet18(pretrained=True)
resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
resnet = nn.Sequential(*list(resnet.children())[:-1])  # remove classifier
resnet.to(device)
resnet.eval()

Sequential(
  (0): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Con

### CNN Embedding Extraction

In [8]:
cell_coords = adata_hvg.obs[['center_x', 'center_y']].copy()
cell_embeddings = []

for _, row in tqdm(cell_coords.iterrows(), total=cell_coords.shape[0], desc="Extracting CNN embeddings"):
    x_px, y_px = microns_to_pixels(row['center_x'], row['center_y'], microns_per_pixel)
    
    # Extract all z-slices
    patches = [extract_patch(z_file, x_px, y_px, patch_size) for z_file in Z_SLICE_FILES]
    
    # Select patch with max variance
    best_idx = np.argmax([np.var(p) for p in patches])
    patch_best = patches[best_idx]
    
    # Resize to patch_size x patch_size
    patch_resized = resize(patch_best, (patch_size, patch_size), preserve_range=True).astype(np.float32)
    
    # Convert to tensor and add channel dimension
    patch_tensor = torch.tensor(patch_resized, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
    
    with torch.no_grad():
        emb = resnet(patch_tensor)
        emb = emb.view(-1).cpu().numpy()
        cell_embeddings.append(emb)

Extracting CNN embeddings: 100%|██████████| 63072/63072 [28:49<00:00, 36.48it/s]  


### Embedding Storage

In [9]:
cell_embeddings = np.stack(cell_embeddings, axis=0)
print("Cell embeddings shape:", cell_embeddings.shape)

# Save to .npy
np.save(EMBEDDINGS_SAVE_PATH, cell_embeddings)
print("CNN embeddings saved to:", EMBEDDINGS_SAVE_PATH)

# Add to AnnData
adata_hvg.obsm['X_cnn'] = cell_embeddings
adata_hvg.write(ADATA_SAVE_PATH)
print("HVG AnnData with CNN embeddings saved to:", ADATA_SAVE_PATH)

Cell embeddings shape: (63072, 512)
CNN embeddings saved to: C:\Users\mills\Desktop\Angione Lab\OC\spatial\OC_adata_hvg_cnn_embeddings.npy
HVG AnnData with CNN embeddings saved to: C:\Users\mills\Desktop\Angione Lab\OC\spatial\OC_adata_hvg_with_cnn.h5ad
